# Notebook 1: Kham pha Du lieu (EDA)

Phan tich tong quan tap du lieu tin tuc cong nghe da scrape.

In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.storage import DataStorage
from config.settings import DB_PATH, RAW_DIR

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
print('Libraries loaded')

## 1. Tai du lieu

In [ ]:
storage = DataStorage(db_path=DB_PATH, raw_dir=RAW_DIR)
df = storage.load_from_sqlite()
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('=== THONG TIN TAP DU LIEU ===')
print(f'Tong so bai:   {len(df)}')
print(f'So cot:        {len(df.columns)}')
print(f'Cot:           {list(df.columns)}')
print()
df.dtypes

## 2. Kiem tra du lieu thieu

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Thieu': missing, 'Ty le (%)': missing_pct})
missing_df[missing_df['Thieu'] > 0].sort_values('Thieu', ascending=False)

## 3. Phan phoi theo nguon bao

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

src_counts = df['source'].value_counts()
axes[0].bar(src_counts.index, src_counts.values, color=['#667eea', '#764ba2'])
axes[0].set_title('So bai theo nguon', fontsize=14, fontweight='bold')
axes[0].set_ylabel('So bai')
for i, v in enumerate(src_counts.values):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

axes[1].pie(src_counts.values, labels=src_counts.index, autopct='%1.1f%%',
            colors=['#667eea', '#764ba2'], startangle=90)
axes[1].set_title('Ty le theo nguon', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Phan phoi theo danh muc

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
cat_counts = df['category'].value_counts()
bars = ax.barh(cat_counts.index, cat_counts.values,
               color=sns.color_palette('husl', len(cat_counts)))
ax.set_xlabel('So bai viet', fontsize=12)
ax.set_title('Phan phoi bai theo danh muc', fontsize=14, fontweight='bold')
for bar, val in zip(bars, cat_counts.values):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2, str(val), va='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Top 10 tac gia

In [ ]:
top_authors = df[df['author'] != 'Khong ro']['author'].value_counts().head(10)
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(top_authors.index, top_authors.values, color=sns.color_palette('viridis', 10))
ax.set_title('Top 10 tac gia nhieu bai nhat', fontsize=14, fontweight='bold')
ax.set_ylabel('So bai')
ax.set_xticklabels(top_authors.index, rotation=45, ha='right')
for i, v in enumerate(top_authors.values):
    ax.text(i, v + 0.1, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Phan phoi do dai bai viet

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['word_count'].dropna(), bins=30, color='#667eea', alpha=0.8, edgecolor='white')
axes[0].set_title('Phan phoi so tu', fontsize=13, fontweight='bold')
axes[0].set_xlabel('So tu')
axes[0].set_ylabel('So bai')

df.boxplot(column='word_count', by='source', ax=axes[1])
axes[1].set_title('So tu theo nguon', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Nguon')
axes[1].set_ylabel('So tu')
plt.suptitle('')

plt.tight_layout()
plt.show()

print('Thong ke so tu:')
print(df['word_count'].describe().round(2))